# Notebook 04 — Open-Meteo Weather API Integration & Offline Fallback Client
**Component 03: Hybrid Explainable Contamination Risk Prediction Framework**

### Research Purpose
Integrates external meteorological context via Open-Meteo APIs (Forecast & Historical Reanalysis). Implements persistent local caching to prevent redundant API hits and provides a guaranteed offline fallback path so internet connectivity is never a single point of failure.

### 🔗 Linked Source Modules & Core Implementations
| Source Module | Core Classes / Functions | Primary Responsibility |
| :--- | :--- | :--- |
| [`src/weather_client.py`](../src/weather_client.py) | `OpenMeteoWeatherClient`, `get_hourly_forecast()`, `fetch_forecast()` | Communicates with Open-Meteo Forecast API with persistent local disk cache and deterministic offline synthetic fallback. |
| [`src/utils.py`](../src/utils.py) | `get_project_root()`, `load_project_config()` | Supplies API endpoints, timeouts, and cache expiry parameters from `project_config.json`. |


In [1]:
# =========================================================================
# 1. ENVIRONMENT SETUP & LINKED WEATHER CLIENT INITIALIZATION
# Linked Module: src/weather_client.py (OpenMeteoWeatherClient)
# =========================================================================
import sys
from pathlib import Path
import pandas as pd

cur = Path.cwd().resolve()
root_dir = cur if (cur / 'config' / 'project_config.json').exists() else cur.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.weather_client import OpenMeteoWeatherClient
# Initialize client with configured cache path in data/external/
client = OpenMeteoWeatherClient()


In [2]:
# =========================================================================
# 2. QUERY WEATHER FORECAST FOR MONITORING CATCHMENT (COLOMBO)
# Linked Method: client.get_hourly_forecast() from src/weather_client.py
# =========================================================================
lat, lon = 6.9271, 79.8612
weather_data = client.get_hourly_forecast(latitude=lat, longitude=lon, forecast_days=3)

print('Weather Client Query Complete!')
print(f'Data Source: {weather_data["source"]}')
print(f'Location: Lat {weather_data["latitude"]}, Lon {weather_data["longitude"]}')
print(f'Timezone: {weather_data["timezone"]}')
print(f'Number of hourly forecasts returned: {len(weather_data["hourly_data"])}')

# Display first 5 forecast steps
pd.DataFrame(weather_data["hourly_data"]).head()


Weather Client Query Complete!
Data Source: disk_cache
Location: Lat 6.9271, Lon 79.8612
Timezone: Asia/Colombo
Number of hourly forecasts returned: 72


,timestamp,weather_temp_c,weather_humidity_pct,rainfall_mm,rain,precipitation_probability_pct,wind_speed_kmh,weather_code,weather_data_type
0,2026-10-18 00:00:00,24.6,98,0.1,0.1,64,6.3,51,disk_cache
1,2026-10-18 01:00:00,24.4,98,0.0,0.0,40,5.0,2,disk_cache
2,2026-10-18 02:00:00,24.4,96,0.0,0.0,22,3.6,3,disk_cache
3,2026-10-18 03:00:00,24.3,96,0.0,0.0,17,3.9,3,disk_cache
4,2026-10-18 04:00:00,24.3,96,0.0,0.0,18,4.4,3,disk_cache


### Weather Integration Summary
The client successfully retrieves precipitation, temperature, relative humidity, and wind speed. Responses are cached in `data/external/`. If the machine goes offline, cached data or synthetic weather fallback is automatically invoked.
